# DDL: `dbspend360_total_pool_spends`

Creates the final per-pool / per-cluster / per-day spend rollup for instance pools, scoped to billing rows
where `usage_metadata.instance_pool_id IS NOT NULL` (any `cluster_source`).

Sibling of `dbspend360_total_job_spends` and `dbspend360_total_all_purpose_spends`, keyed on
`(instance_pool_id, cluster_id, usage_date)`. Pool metadata (`pool_name`, `node_type`,
`min_idle_instances`, `max_capacity`, `idle_instance_autotermination_minutes`) is denormalized from
`system.compute.instance_pools` so the UI does not need a live join.

Three-state snapshot handling (per the plan §3.5):
- Active pool: `pool_snapshot_missing = FALSE`, `pool_deleted_at IS NULL`.
- Deleted but visible: `pool_snapshot_missing = FALSE`, `pool_deleted_at` populated
  (UI renders "Deleted YYYY-MM-DD" badge).
- Snapshot missing entirely: `pool_snapshot_missing = TRUE`, `pool_deleted_at IS NULL`
  (UI renders "Snapshot missing" badge; `pool_name` falls back to `"Pool {instance_pool_id}"`).

v1 ships without cloud cost (idle pool capacity is structurally invisible to `system.billing.usage`).
`cloud_cost DOUBLE` is reserved but always NULL in v1; `total_cost = databricks_cost + COALESCE(cloud_cost, 0)`
so v2 can populate `cloud_cost` without a schema migration.

**No `pool_creator_id` column.** `system.compute.instance_pools.tags` is documented as user-defined tags
only and excludes default tags, so the auto-applied `DatabricksInstancePoolCreatorId` is not visible from
the system table. Creator info is resolved per-request in the pool details modal via the REST API
(`WorkspaceClient.instance_pools.get(...).default_tags`).

**Widgets**
- `catalog` - target Unity Catalog name
- `schema`  - target schema name within `catalog`

In [ ]:
dbutils.widgets.text("catalog", "", "Catalog")
dbutils.widgets.text("schema", "", "Schema")

In [ ]:
catalog = dbutils.widgets.get("catalog").strip()
schema = dbutils.widgets.get("schema").strip()

if not catalog or not schema:
    raise ValueError("Both `catalog` and `schema` widgets must be set.")

spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")

In [ ]:
%sql
CREATE TABLE IF NOT EXISTS ${catalog}.${schema}.dbspend360_total_pool_spends (
  instance_pool_id                       STRING,
  cluster_id                             STRING,
  usage_date                             DATE,
  workspace_id                           STRING,
  pool_name                              STRING,
  node_type                              STRING,
  min_idle_instances                     BIGINT,
  max_capacity                           BIGINT,
  idle_instance_autotermination_minutes  BIGINT,
  pool_snapshot_missing                  BOOLEAN,
  pool_deleted_at                        TIMESTAMP,
  databricks_cost                        DOUBLE,
  cloud_cost                             DOUBLE,
  total_cost                             DOUBLE,
  currency                               STRING,
  sku_name                               STRING,
  created_at                             TIMESTAMP,
  updated_at                             TIMESTAMP
)
CLUSTER BY AUTO

In [ ]:
dbutils.notebook.exit(f"{catalog}.{schema}.dbspend360_total_pool_spends")